# Only Segments (no images)


In [ ]:
#segment only on map
import pandas as pd
import geopandas as gpd
from shapely import wkt
import folium


df = pd.read_csv("../Scores/all_scored_edges_filtered_with_ai.csv")
df["geometry"] = df["geometry"].apply(wkt.loads)

edges = gpd.GeoDataFrame(
    df,
    geometry="geometry",
    crs="EPSG:4326"
)

# boundary box so only load the edges necessary

north = 47.61758024981437
south = 47.613100153406556
east  = -122.32635157491734
west  = -122.33482914351927
edges_area = edges.cx[west:east, south:north]

# create map
m = folium.Map(
    location=[
        (north + south) / 2,
        (east + west) / 2
    ],
    zoom_start=17,
    tiles="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors"
)

colors = [
    "#0aa519"
]

# add edges
for i, (_, row) in enumerate(edges_area.iterrows()):

    # geometry coordinates are lon,lat
    coords = [
        [lat, lon]
        for lon, lat in row.geometry.coords
    ]

    folium.PolyLine(
        coords,
        color=colors[i % len(colors)],
        weight=6,
        opacity=0.9
    ).add_to(m)
    
    # mark the segment midpoints so can differentiate them
    midpoint = row.geometry.interpolate(0.5, normalized=True)
    folium.CircleMarker(
      location=[midpoint.y, midpoint.x],
      radius=5,          
      color="black",     
      weight=1,
      fill=True,
      fill_color="white",    
      fill_opacity=1,
      popup=folium.Popup(
          f"""
          OSM ID: {row['osm_id']}<br>
          u: {row['u']}<br>
          v: {row['v']}
          """,
          max_width=300
      )
    ).add_to(m)
 
    origin = [47.61671222562014, -122.33311978530625]
    destination = [47.61575302975626, -122.32800279032077]
   
   
    folium.Marker(
         origin,
         popup="Origin",
         icon=folium.Icon(color="green", icon="play")
    ).add_to(m)
 
   
  

# Display map
m.save("seattle_edges_map.html")

/var/folders/p7/5hrm9t_12rs2c4jgh750nc200000gn/T/ipykernel_61567/3862026418.py:7: DtypeWarning: Columns (0: maxspeed) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../Scores/all_scored_edges_filtered_with_ai.csv")


# Mapping segments with images

In [ ]:
import folium
import geopandas as gpd
import pandas as pd
from shapely import wkt
from shapely.geometry import Point

df = pd.read_csv("../Scores/all_scored_edges_filtered_with_ai.csv")
df["geometry"] = df["geometry"].apply(wkt.loads)
images_df = pd.read_csv("../Scores/combined_svi_data.csv")

edges = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

# boundary box 
north = 47.61758024981437
south = 47.613100153406556
east = -122.32635157491734
west = -122.33482914351927

edges_area = edges.cx[west:east, south:north].copy()
edges_area["orig_idx"] = edges_area.index


# to add buffer
projected_crs = "EPSG:32610"
edges_proj = edges_area.to_crs(projected_crs)
edges_proj["buffer"] = edges_proj.geometry.buffer(10) # currently 10 m buffer 

buffered_edges = gpd.GeoDataFrame(
    edges_proj, geometry="buffer", crs=projected_crs
)

# images into gdf
images_df["geometry"] = images_df.apply(
    lambda row: Point(row["Longitude"], row["Latitude"]), axis=1
)
images_gdf = gpd.GeoDataFrame(images_df, geometry="geometry", crs="EPSG:4326")
images_proj = images_gdf.to_crs(projected_crs)

# get images within 10 m of street segment lat/longs
joined = gpd.sjoin(images_proj, buffered_edges, how="inner", predicate="within")

# group together
segment_images_dict = (
    joined.groupby("orig_idx")
    .agg(image_list=("ImageID", list))
    .to_dict()["image_list"]
)

# only do the images within the bbox (obviously)
images_area = images_df[
    (images_df["Latitude"] >= south)
    & (images_df["Latitude"] <= north)
    & (images_df["Longitude"] >= west)
    & (images_df["Longitude"] <= east)
]

# making map
m = folium.Map(
    location=[(north + south) / 2, (east + west) / 2],
    zoom_start=17,
    tiles="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors",
)

colors = ["#0aa519"]

# map details
for i, (idx, row) in enumerate(edges_area.iterrows()):
    # Fetch matched images using the tracked row index
    nearby_imgs = segment_images_dict.get(idx, [])
    imgs_str = ", ".join(nearby_imgs) if nearby_imgs else "None"

    coords = [[lat, lon] for lon, lat in row.geometry.coords]

    folium.PolyLine(
        coords,
        color=colors[i % len(colors)],
        weight=6,
        opacity=0.9
    ).add_to(m)

    # segment markers
    midpoint = row.geometry.interpolate(0.5, normalized=True)
    folium.CircleMarker(
        location=[midpoint.y, midpoint.x],
        radius=5,
        color="black",
        weight=1,
        fill=True,
        fill_color="white",
        fill_opacity=1,
        popup=folium.Popup(
            f"""
          OSM ID: {row['osm_id']}<br>
          u: {row['u']}<br>
          v: {row['v']}<br>
          <b>Nearby Images:</b> {imgs_str}
          """,
            max_width=300,
        ),
    ).add_to(m)

# image markers
for _, img_row in images_area.iterrows():
    folium.CircleMarker(
        location=[img_row["Latitude"], img_row["Longitude"]],
        radius=3,
        color="red",
        weight=1,
        fill=True,
        fill_color="red",
        fill_opacity=0.8,
        tooltip=str(img_row["ImageID"]),
        popup=f"Image ID: {img_row['ImageID']}",
    ).add_to(m)

# origin and destination markers
origin = [47.61671222562014, -122.33311978530625]
destination = [47.61575302975626, -122.32800279032077]

folium.Marker(
    origin, popup="Origin", icon=folium.Icon(color="green", icon="play")
).add_to(m)

folium.Marker(
    destination,
    popup="Destination",
    icon=folium.Icon(color="red", icon="stop"),
).add_to(m)

# saving
m.save("seattle_edges_map_with_images.html")

/var/folders/p7/5hrm9t_12rs2c4jgh750nc200000gn/T/ipykernel_66127/2130683003.py:7: DtypeWarning: Columns (0: maxspeed) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../Scores/all_scored_edges_filtered_with_ai.csv")


Map successfully saved with working segment-to-image hover data!


# Putting images per segment (per path) into csv

In [ ]:
# Segment-level image lists
import ast
import geopandas as gpd
import pandas as pd
from shapely import wkt
from shapely.geometry import Point

# 1. Load the data using your exact paths
paths_df = pd.read_csv("paths.csv")
edges_df = pd.read_csv("../Scores/all_scored_edges_filtered_with_ai.csv")
images_df = pd.read_csv("../Scores/combined_svi_data.csv")

# explodings egments
paths_df["segments"] = paths_df["segments_json"].apply(ast.literal_eval)
exploded_paths = paths_df.explode("segments").reset_index(drop=True)

# Extract edge data
exploded_paths["u"] = exploded_paths["segments"].apply(lambda x: str(x["u"]))
exploded_paths["v"] = exploded_paths["segments"].apply(lambda x: str(x["v"]))
exploded_paths["osm_ids"] = exploded_paths["segments"].apply(
    lambda x: str(x["osm_ids"])
)

# clean edges
edges_df["u"] = edges_df["u"].astype(str)
edges_df["v"] = edges_df["v"].astype(str)
edges_df["geometry"] = edges_df["geometry"].apply(wkt.loads)
edges_gdf = gpd.GeoDataFrame(edges_df, geometry="geometry", crs="EPSG:4326")

# merge edges
path_segments = pd.merge(
    exploded_paths[["path_id", "u", "v", "osm_ids"]],
    edges_gdf[["u", "v", "geometry"]],
    on=["u", "v"],
    how="inner",
)
path_segments_gdf = gpd.GeoDataFrame(path_segments, geometry="geometry", crs="EPSG:4326")

# images to gdf
images_df["geometry"] = images_df.apply(
    lambda row: Point(row["Longitude"], row["Latitude"]), axis=1
)
images_gdf = gpd.GeoDataFrame(images_df, geometry="geometry", crs="EPSG:4326")

# for buffer again
projected_crs = "EPSG:32610"
path_segments_proj = path_segments_gdf.to_crs(projected_crs)
images_proj = images_gdf.to_crs(projected_crs)
path_segments_proj["buffer"] = path_segments_proj.geometry.buffer(10) # 10 m buffer
buffered_segments = gpd.GeoDataFrame(
    path_segments_proj[["path_id", "u", "v", "osm_ids"]],
    geometry=path_segments_proj["buffer"],
    crs=projected_crs,
)

result = gpd.sjoin(images_proj, buffered_segments, how="inner", predicate="within")
unique_segment_images = result.drop_duplicates(subset=["path_id", "u", "v", "ImageID"])

final_output = (
    unique_segment_images.groupby(["path_id", "osm_ids", "u", "v"])
    .agg(image_list=("ImageID", list))
    .reset_index()
)
final_output = final_output[["path_id", "osm_ids", "u", "v", "image_list"]]

# save
output_filename = "paths_with_images.csv"
final_output.to_csv(output_filename, index=False)
print(f"Successfully generated '{output_filename}' containing segment-level image lists!")

/var/folders/p7/5hrm9t_12rs2c4jgh750nc200000gn/T/ipykernel_66127/1281389041.py:9: DtypeWarning: Columns (0: maxspeed) have mixed types. Specify dtype option on import or set low_memory=False.
  edges_df = pd.read_csv("../Scores/all_scored_edges_filtered_with_ai.csv")


Successfully generated 'paths_with_images.csv' containing segment-level image lists!
